In [ ]:
#!/usr/bin/env python3
import os
import random
import pickle
import argparse
import time
import json
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Tuple
try:
    import psutil
except Exception:
    psutil = None
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.multioutput import MultiOutputRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import lightgbm as lgb

SEED = 12345
random.seed(SEED)
np.random.seed(SEED)

METRICS_LOG = "farm_metrics.jsonl"
CONSOLIDATED_BEST = "farm_best_clis_all.txt"
CONSOLIDATED_ALT = "farm_alt_clis_all.txt"
CONSOLIDATED_PF = "farm_configs.npy"

if os.path.exists(METRICS_LOG):
    os.remove(METRICS_LOG)

def append_metrics(row: dict):
    with open(METRICS_LOG, "a") as f:
        f.write(json.dumps(row, default=lambda x: None) + "\n")

df_raw = pd.read_excel("farm_data.xlsx")
df = df_raw.dropna(subset=["energy", "time"]).copy()

max_workers = int(df["number_of_workers"].max())
for i in range(1, max_workers + 1):
    df[f"workload_size_w{i}"] = df.get(f"workload_size_w{i}", 0).fillna(0).astype(int)
    df[f"w{i}_threads"] = df.get(f"w{i}_threads", 0).fillna(0).astype(int)

df["workload_type"] = df["workload_type"].astype(str).fillna("unknown")
df["core_binding_type"] = df.get("core_binding_type", "manual_roundrobin").astype(str).fillna("manual_roundrobin")
df["workload_balance"] = df.get("workload_balance", "unknown").astype(str).fillna("unknown")

cpu_cols = [c for c in df.columns if c.startswith("cpu") and c.endswith("_freq")]
little = [c for c in cpu_cols if int(''.join(filter(str.isdigit, c))) < 4]
big = [c for c in cpu_cols if int(''.join(filter(str.isdigit, c))) >= 4]

df["avg_little_freq"] = df[little].mean(axis=1) if little else 1_500_000
df["avg_big_freq"] = df[big].mean(axis=1) if big else 2_000_000
df["total_cores_on"] = df["big_cores_on"] + df["little_cores_on"]
df["imbalance"] = abs(df["big_cores_on"] - df["little_cores_on"])
df["ratio_big_to_little"] = df["big_cores_on"] / (df["little_cores_on"] + 1e-6)
df["total_processing_threads"] = df[[f"w{i}_threads" for i in range(1, max_workers + 1)]].sum(axis=1)

categorical_cols = ["workload_type", "core_binding_type", "workload_balance"]
numeric_cols = [
    "number_of_workers", "chunk_size", "total_processing_threads",
    "big_cores_on", "little_cores_on", "total_cores_on",
    "imbalance", "ratio_big_to_little",
    "avg_little_freq", "avg_big_freq"
] + [f"workload_size_w{i}" for i in range(1, max_workers + 1)] + [f"w{i}_threads" for i in range(1, max_workers + 1)]

X = pd.concat([df[categorical_cols].astype(str), df[numeric_cols]], axis=1)
y = df[["energy", "time"]]

preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="mean"), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

lgb_params = {"n_estimators": 50, "random_state": SEED, "n_jobs": -1}
model_farm = Pipeline([
    ("prep", preprocessor),
    ("rf", MultiOutputRegressor(lgb.LGBMRegressor(**lgb_params)))
])

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.15, random_state=SEED)

def fit_with_metrics(pipe: Pipeline, X_train: pd.DataFrame, y_train: pd.DataFrame) -> Tuple[Pipeline, float, int]:
    start = time.perf_counter()
    peak_mem = -1
    if psutil is None:
        pipe.fit(X_train, y_train)
        train_time = time.perf_counter() - start
        return pipe, train_time, -1
    proc = psutil.Process()
    import threading
    done = threading.Event()
    mem_samples = []
    def sampler():
        while not done.is_set():
            try:
                mem_samples.append(proc.memory_info().rss)
            except Exception:
                pass
            time.sleep(0.01)
    thread = threading.Thread(target=sampler, daemon=True)
    thread.start()
    try:
        pipe.fit(X_train, y_train)
    finally:
        done.set()
        thread.join(timeout=1.0)
    train_time = time.perf_counter() - start
    peak_mem = int(max(mem_samples) if mem_samples else -1)
    return pipe, train_time, peak_mem

model_farm, train_time_s, peak_mem_bytes = fit_with_metrics(model_farm, Xtr, ytr)

preds = model_farm.predict(Xte)
rmse_e = np.sqrt(((preds[:,0] - yte.energy)**2).mean())
rmse_t = np.sqrt(((preds[:,1] - yte.time)**2).mean())
mae_e = mean_absolute_error(yte["energy"], preds[:,0])
mae_t = mean_absolute_error(yte["time"], preds[:,1])
r2_e = r2_score(yte["energy"], preds[:,0])
r2_t = r2_score(yte["time"], preds[:,1])

print(f"Surrogate trained → RMSE energy={rmse_e:.2f}, time={rmse_t:.2f}")
print(f"MAE           → energy={mae_e:.2f}, time={mae_t:.2f}")
print(f"R2            → energy={r2_e:.3f}, time={r2_t:.3f}")
print(f"Train time (s): {train_time_s:.3f}, Peak mem (bytes): {peak_mem_bytes}")

baseline_mae_e = float(np.mean(np.abs(yte["energy"] - yte["energy"].mean())))
baseline_mae_t = float(np.mean(np.abs(yte["time"] - yte["time"].mean())))
yte_std_energy = float(yte["energy"].std()); yte_std_time = float(yte["time"].std())
r2_reliable = (yte_std_energy > 1e-6) and (yte_std_time > 1e-6)
print("baseline MAE energy:", baseline_mae_e, "time:", baseline_mae_t)
print("yte std energy:", yte_std_energy, "time:", yte_std_time, "r2_reliable:", r2_reliable)

tmp_model_file = "farm_model_tmp.pkl"
with open(tmp_model_file, "wb") as f:
    pickle.dump(model_farm, f, protocol=pickle.HIGHEST_PROTOCOL)
model_size_bytes = os.path.getsize(tmp_model_file)
os.remove(tmp_model_file)
print(f"Model serialized size (bytes): {model_size_bytes}")

def predict_perf(pipe: Pipeline, Xc: pd.DataFrame, reps: int = 5) -> Tuple[float, float, np.ndarray]:
    _ = pipe.predict(Xc.iloc[:min(10, len(Xc))])
    times = []
    preds_local = None
    for _ in range(reps):
        t0 = time.perf_counter()
        preds_local = pipe.predict(Xc)
        t1 = time.perf_counter()
        times.append((t1 - t0) / max(1, len(Xc)))
    return float(np.median(times)), float(np.std(times)), preds_local

pred_latency_per_sample_s, pred_latency_std_s, _ = predict_perf(model_farm, Xte, reps=5)
print(f"Prediction latency per sample: {pred_latency_per_sample_s*1e3:.3f} ms (std {pred_latency_std_s*1e3:.3f} ms)")

pd.concat([Xte.reset_index(drop=True).iloc[:100],
           pd.DataFrame(preds, columns=["pred_energy","pred_time"]).iloc[:100].reset_index(drop=True),
           yte.reset_index(drop=True).iloc[:100].reset_index(drop=True)], axis=1).to_csv("Xte_sample_pred_truth_farm.csv", index=False)
print("Wrote sample diagnostics to Xte_sample_pred_truth_farm.csv")

imp = model_farm.named_steps["prep"].named_transformers_["num"]
num_means = imp.statistics_.tolist()
ohe = model_farm.named_steps["prep"].named_transformers_["cat"]
categories = [list(c) for c in ohe.categories_]

def lgb_tree_to_dict(tree_struct):
    if "leaf_value" in tree_struct:
        return {"leaf": True, "value": float(tree_struct["leaf_value"])}
    feature = int(tree_struct.get("split_feature", -1))
    threshold = float(tree_struct.get("threshold", 0.0))
    left = lgb_tree_to_dict(tree_struct["left_child"])
    right = lgb_tree_to_dict(tree_struct["right_child"])
    return {"leaf": False, "feature": feature, "threshold": threshold, "left": left, "right": right}

def lgb_model_to_forest(estimator):
    booster = estimator.booster_
    dump = booster.dump_model()
    trees = []
    for t in dump.get("tree_info", []):
        tree_struct = t.get("tree_structure", {})
        trees.append(lgb_tree_to_dict(tree_struct))
    return trees

mor = model_farm.named_steps["rf"]
forests = []
for est in mor.estimators_:
    try:
        trees = lgb_model_to_forest(est)
    except Exception:
        try:
            trees = lgb_model_to_forest(est)
        except Exception:
            trees = []
    forests.append(trees)

spec = {"num_cols": numeric_cols, "num_means": num_means,
        "cat_cols": categorical_cols, "categories": categories,
        "forests": forests, "alpha": [1.0, 1.0], "beta": [0.0, 0.0]}

with open("farm_model_numpy.pkl", "wb") as f:
    pickle.dump(spec, f, protocol=pickle.HIGHEST_PROTOCOL)

with open("farm_columns.txt", "w") as f:
    for col in categorical_cols + numeric_cols:
        f.write(col + "\n")
print("Saved farm_model_numpy.pkl and farm_columns.txt")

core_combos = [
    (int(b), int(l))
    for b in df["big_cores_on"].unique()
    for l in df["little_cores_on"].unique()
    if (b + l) > 0
]
freq_l = np.unique(df[little].values.ravel()) if little else np.array([1_500_000])
freq_b = np.unique(df[big].values.ravel()) if big else np.array([2_000_000])
freq_samples = [(np.random.choice(freq_l, 4), np.random.choice(freq_b, 4)) for _ in range(500)]
thread_totals = df["total_processing_threads"].unique()
chunk_sizes = df["chunk_size"].unique() if "chunk_size" in df else np.array([1])
wbalance_domain = df["workload_balance"].astype(str).unique().tolist()

def pareto_mask(E: np.ndarray, T: np.ndarray) -> np.ndarray:
    keep = np.ones(len(E), dtype=bool)
    for i in range(len(E)):
        if not keep[i]:
            continue
        dom = (E <= E[i]) & (T <= T[i]) & ((E < E[i]) | (T < T[i]))
        dom[i] = False
        if dom.any():
            keep[i] = False
    return keep

def random_candidates(workload_type: str, workload_size: list, number_of_workers: int, n_candidates: int = 2000):
    wl = np.atleast_1d(workload_size)
    if wl.size == 1:
        wl = np.full(number_of_workers, wl.item(), dtype=int)

    mask = (df["workload_type"] == workload_type) & (df["number_of_workers"] == number_of_workers)
    for i in range(1, number_of_workers + 1):
        col = f"workload_size_w{i}"
        ser = pd.to_numeric(df.get(col, pd.Series(dtype=float)), errors="coerce").fillna(-1).astype(int)
        mask &= (ser == wl[i - 1])
    real_rows = df[mask]

    valid_totals = list(thread_totals) if len(thread_totals) else [number_of_workers]
    rows = []
    for _ in range(n_candidates):
        b_on, l_on = random.choice(core_combos)
        lf, bf = random.choice(freq_samples)
        avg_l = float(np.mean(lf)); avg_b = float(np.mean(bf))
        thr_total = int(random.choice(valid_totals))
        base = np.ones(number_of_workers, dtype=int)
        leftover = max(0, thr_total - number_of_workers)
        extra = np.random.multinomial(leftover, [1/number_of_workers] * number_of_workers)
        alloc = base + extra
        chunk = int(random.choice(chunk_sizes))
        wb = random.choice(wbalance_domain)
        entry = {
            "workload_type": workload_type, "number_of_workers": number_of_workers,
            "workload_balance": wb, "chunk_size": chunk, "core_binding_type": "manual_roundrobin",
            "big_cores_on": int(b_on), "little_cores_on": int(l_on),
            "total_cores_on": int(b_on + l_on), "imbalance": abs(int(b_on - l_on)),
            "ratio_big_to_little": float(b_on / (l_on + 1e-6)),
            "avg_little_freq": avg_l, "avg_big_freq": avg_b,
            "total_processing_threads": thr_total
        }
        for i in range(1, number_of_workers + 1):
            entry[f"workload_size_w{i}"] = int(wl[i - 1])
            entry[f"w{i}_threads"] = int(alloc[i - 1])
        for i in range(number_of_workers + 1, max_workers + 1):
            entry[f"workload_size_w{i}"] = 0
            entry[f"w{i}_threads"] = 0
        rows.append(entry)

    for _, r in real_rows.iterrows():
        cand = {k: r[k] for k in rows[0].keys() if k in r.index}
        rows.append(cand)

    return pd.DataFrame(rows), int(len(real_rows))

def build_cli(row: pd.Series) -> str:
    b_on = int(row["big_cores_on"]); l_on = int(row["little_cores_on"])
    lf = row["avg_little_freq"] / 1e6; bf = row["avg_big_freq"] / 1e6
    workers = int(row["number_of_workers"]); wt = row["workload_type"]; chunk = int(row["chunk_size"])
    sizes = " ".join(str(int(row[f"workload_size_w{i}"])) for i in range(1, workers + 1))
    thrs = " ".join(str(int(row[f"w{i}_threads"])) for i in range(1, workers + 1))
    return (f"--big-cores {b_on} --little-cores {l_on} "
            f"--little-freq {lf:.1f}GHz --big-freq {bf:.1f}GHz "
            f"./farm {workers} {wt} {sizes} {chunk} {thrs}")

def recommend_all(workload_type: str, workload_size: list, number_of_workers: int, n_candidates: int = 2000, K: int = 5):
    cand_df, real_count = random_candidates(workload_type, workload_size, number_of_workers, n_candidates)
    if real_count == 0:
        print("No exact-match real runs; using ML for all candidates")

    Xc = pd.concat([cand_df[categorical_cols].astype(str), cand_df[numeric_cols]], axis=1)
    pred_lat_s, pred_lat_std_s, preds = predict_perf(model_farm, Xc, reps=3)
    cand_df["energy_pred"], cand_df["time_pred"] = preds[:,0], preds[:,1]

    calib = Path("farm_calib.csv")
    if calib.exists():
        df_cal = pd.read_csv(calib)
        Xcal = pd.concat([df_cal[categorical_cols].astype(str), df_cal[numeric_cols]], axis=1)
        pcal = model_farm.predict(Xcal)
        lr_e = LinearRegression().fit(pcal[:,[0]], df_cal["energy_real"])
        lr_t = LinearRegression().fit(pcal[:,[1]], df_cal["time_real"])
        cand_df["energy_pred"] = lr_e.predict(cand_df[["energy_pred"]])
        cand_df["time_pred"] = lr_t.predict(cand_df[["time_pred"]])
        print("Applied calibration from farm_calib.csv")
    else:
        print("farm_calib.csv not found; skipping calibration")

    mask = pareto_mask(cand_df["energy_pred"].to_numpy(), cand_df["time_pred"].to_numpy())
    pf = cand_df[mask].copy()
    pf["balanced_pred"] = pf["energy_pred"]/pf["energy_pred"].max() + pf["time_pred"]/pf["time_pred"].max()

    np.save("farm_configs.npy", pd.concat([pf[categorical_cols].astype(str), pf[numeric_cols]], axis=1).to_numpy())
    print("Saved farm_configs.npy")

    row_e = pf.nsmallest(1, "energy_pred").iloc[0]
    row_t = pf.nsmallest(1, "time_pred").iloc[0]
    row_b = pf.nsmallest(1, "balanced_pred").iloc[0]
    results = {"energy": build_cli(row_e), "time": build_cli(row_t), "balanced": build_cli(row_b)}

    def top_alts(df, col):
        df2 = df.copy()
        best_idx = df2[col].idxmin()
        alts = df2.nsmallest(K+1, col).drop(best_idx)
        seen, uniq = set(), []
        for _, r in alts.iterrows():
            cli = build_cli(r)
            if cli in seen:
                continue
            seen.add(cli); uniq.append(cli)
            if len(uniq) == K:
                break
        return uniq

    alts = {"energy": top_alts(pf, "energy_pred"), "time": top_alts(pf, "time_pred"), "balanced": top_alts(pf, "balanced_pred")}

    pf_keys = set(build_cli(r) for _, r in pf.iterrows())
    topk_energy = set(build_cli(r) for _, r in pf.nsmallest(K+1, "energy_pred").drop(pf.nsmallest(1, "energy_pred").index).iterrows())
    topk_time = set(build_cli(r) for _, r in pf.nsmallest(K+1, "time_pred").drop(pf.nsmallest(1, "time_pred").index).iterrows())
    topk_bal = set(build_cli(r) for _, r in pf.nsmallest(K+1, "balanced_pred").drop(pf.nsmallest(1, "balanced_pred").index).iterrows())

    metrics_row = {
        "timestamp": time.time(), "seed": SEED,
        "workload_type": workload_type, "workload_sizes": list(map(int, workload_size)),
        "number_of_workers": number_of_workers, "n_candidates": len(cand_df),
        "train_time_s": float(train_time_s), "peak_mem_bytes": int(peak_mem_bytes), "model_size_bytes": int(model_size_bytes),
        "pred_latency_per_sample_ms_candidates": float(pred_lat_s*1e3), "pred_latency_std_ms_candidates": float(pred_lat_std_s*1e3),
        "pareto_count": int(len(pf_keys)),
        "best_energy_cli": results["energy"], "best_time_cli": results["time"], "best_balanced_cli": results["balanced"],
        "topk_energy": sorted(list(topk_energy)), "topk_time": sorted(list(topk_time)), "topk_balanced": sorted(list(topk_bal)),
        "baseline_mae_energy": baseline_mae_e, "baseline_mae_time": baseline_mae_t,
        "yte_std_energy": yte_std_energy, "yte_std_time": yte_std_time, "r2_reliable": bool(r2_reliable)
    }

    append_metrics(metrics_row)

    return results, alts, {"pf_keys": pf_keys, "best": {"energy": results["energy"], "time": results["time"], "balanced": results["balanced"]},
                           "topk": {"energy": topk_energy, "time": topk_time, "balanced": topk_bal}}

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Train farm LightGBM surrogate & emit Pareto configs")
    parser.add_argument("--n_candidates", "-c", type=int, default=2000)
    parser.add_argument("--K", "-k", type=int, default=5)
    args, _ = parser.parse_known_args()

    wts = ["cpu_only", "memory_only", "cpu_and_memory", "memory_and_cpu"]
    vectors = {
        "cpu_only": [[1_000_000]*3, [5_000_000]*3, [10_000_000]*3, [1_000_000,2_000_000,3_000_000]],
        "memory_only": [[2_000_000]*4, [4_000_000]*4],
        "cpu_and_memory": [[8_000_000]*2, [16_000_000]*2],
        "memory_and_cpu": [[8_000_000]*2, [16_000_000]*2],
    }

    all_pf = []
    rmse_energy_list = []; rmse_time_list = []
    mae_energy_list = []; mae_time_list = []
    r2_energy_list = []; r2_time_list = []
    consolidated_best_clis = []; consolidated_alt_clis = set()
    combos_processed = 0

    for wt in wts:
        for wl in vectors.get(wt, []):
            num_workers = len(wl)
            results, alts, artifacts = recommend_all(workload_type=wt, workload_size=wl,
                                                    number_of_workers=num_workers,
                                                    n_candidates=args.n_candidates, K=args.K)

            vec_str = "_".join(str(x) for x in wl)
            combo = f"{wt}_{num_workers}workers_{vec_str}"
            print(f"\n=== RESULTS for {combo} ===")
            print("ENERGY   →", results["energy"])
            print("TIME     →", results["time"])
            print("BALANCED →", results["balanced"])
            print(f"\n--- ALTERNATIVES for {combo} ---")
            for obj in ("balanced", "energy", "time"):
                for cli in alts[obj]:
                    print(" ", cli)

            pf_arr = np.load("farm_configs.npy", allow_pickle=True)
            all_pf.append(pf_arr)

            consolidated_best_clis.append({"combo": combo, "best_energy": artifacts["best"]["energy"],
                                           "best_time": artifacts["best"]["time"], "best_balanced": artifacts["best"]["balanced"]})
            for s in artifacts["pf_keys"]:
                consolidated_alt_clis.add(s)
            for objset in artifacts["topk"].values():
                for k in objset:
                    consolidated_alt_clis.add(k)

            rmse_energy_list.append(float(rmse_e)); rmse_time_list.append(float(rmse_t))
            mae_energy_list.append(float(mae_e)); mae_time_list.append(float(mae_t))
            r2_energy_list.append(float(r2_e)); r2_time_list.append(float(r2_t))

            combos_processed += 1
            print(f"Processed {combo}")

    combined = np.vstack(all_pf) if all_pf else np.zeros((0, len(categorical_cols) + len(numeric_cols)))
    np.save(CONSOLIDATED_PF, combined)

    with open(CONSOLIDATED_BEST, "w") as f:
        for entry in consolidated_best_clis:
            f.write(json.dumps(entry) + "\n")
    with open(CONSOLIDATED_ALT, "w") as f:
        for cli in sorted(consolidated_alt_clis):
            f.write(cli + "\n")

    def summarize(lst):
        if not lst:
            return {"mean": None, "std": None, "count": 0}
        a = np.array(lst, dtype=float); return {"mean": float(a.mean()), "std": float(a.std(ddof=0)), "count": int(a.size)}

    totals = {
        "rmse_energy": summarize(rmse_energy_list),
        "rmse_time": summarize(rmse_time_list),
        "mae_energy": summarize(mae_energy_list),
        "mae_time": summarize(mae_time_list),
        "r2_energy": summarize(r2_energy_list),
        "r2_time": summarize(r2_time_list),
        "combos_processed": combos_processed
    }

    rmse_total = (totals["rmse_energy"]["mean"] + totals["rmse_time"]["mean"]) if totals["rmse_energy"]["mean"] is not None else None
    mae_total = (totals["mae_energy"]["mean"] + totals["mae_time"]["mean"]) if totals["mae_energy"]["mean"] is not None else None
    r2_total = (totals["r2_energy"]["mean"] + totals["r2_time"]["mean"]) if totals["r2_energy"]["mean"] is not None else None

    print("\n=== AGGREGATED METRICS ACROSS COMBOS (TOTAL) ===")
    print(f"RMSE energy mean={totals['rmse_energy']['mean']:.3f} std={totals['rmse_energy']['std']:.3f}")
    print(f"RMSE time   mean={totals['rmse_time']['mean']:.3f} std={totals['rmse_time']['std']:.3f}")
    print(f"MAE  energy mean={totals['mae_energy']['mean']:.3f} std={totals['mae_energy']['std']:.3f}")
    print(f"MAE  time   mean={totals['mae_time']['mean']:.3f} std={totals['mae_time']['std']:.3f}")
    print(f"R2   energy mean={totals['r2_energy']['mean']:.6f} std={totals['r2_energy']['std']:.6f}")
    print(f"R2   time   mean={totals['r2_time']['mean']:.6f} std={totals['r2_time']['std']:.6f}")

    print("\n=== EXPLICIT TOTALS (SUM OF ENERGY + TIME MEANS) ===")
    print(f"RMSE total = RMSE_energy_mean + RMSE_time_mean = {rmse_total:.3f}")
    print(f"MAE  total = MAE_energy_mean  + MAE_time_mean  = {mae_total:.3f}")
    print(f"R2   total = R2_energy_mean   + R2_time_mean   = {r2_total:.6f}")

    summary_out = {
        "totals": totals,
        "explicit_totals": {"rmse_total": rmse_total, "mae_total": mae_total, "r2_total": r2_total},
        "global_test": {"rmse_energy": float(rmse_e), "rmse_time": float(rmse_t),
                        "mae_energy": float(mae_e), "mae_time": float(mae_t),
                        "r2_energy": float(r2_e), "r2_time": float(r2_t)},
        "model": {"n_estimators": lgb_params["n_estimators"], "model_size_bytes": model_size_bytes,
                  "train_time_s": train_time_s, "peak_mem_bytes": peak_mem_bytes},
        "notes": "Totals are means across combos; explicit_totals are sums of energy+time means."
    }
    with open("farm_aggregated_summary.json", "w") as f:
        json.dump(summary_out, f, indent=2)

    print(f"\nSaved COMBINED {CONSOLIDATED_PF} ({combined.shape[0]} total configs across all combos)")
    print(f"Wrote consolidated best CLIs → {CONSOLIDATED_BEST}")
    print(f"Wrote consolidated alt CLIs  → {CONSOLIDATED_ALT}")
    print(f"Wrote aggregated summary     → farm_aggregated_summary.json")
    print(f"Per-combo metrics log        → {METRICS_LOG}")
